# Resumo Detalhado dos Dados PPM (Pesquisa Pecuária Municipal)

Este notebook consolida a análise dos rebanhos brasileiros por tipo e município.

In [2]:
import os
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import numpy as np

# Configuração de caminhos dinâmica
# O notebook está em notebooks/etl-dados-necessarios/ppm/data/resume/
BASE_DIR = Path.cwd().parent.parent # notebooks/etl-dados-necessarios/ppm/
PPM_BRONZE_DIR = BASE_DIR / "data/bronze"
OUTPUT_DIR = BASE_DIR / "data/resume"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def analyze_ppm_qualitative(directory):
    # Lista pastas que começam com 'ppm_' na bronze
    datasets = [d for d in directory.iterdir() if d.is_dir() and d.name.startswith('ppm_')]
    if not datasets:
        print(f"❌ Nenhum dataset PPM (ppm_*) encontrado em {directory}.")
        # Fallback para listar o conteúdo e entender a estrutura
        print(f"Conteúdo de {directory}: {[d.name for d in directory.iterdir()]}")
        return None
    
    print(f"🔍 Analisando {len(datasets)} categorias de rebanhos (PPM)...")
    
    all_stats = []
    
    for ds_path in tqdm(datasets, desc="Processando Categorias PPM"):
        ds_name = ds_path.name.replace('ppm_', '').upper() # Ex: BOVINOS
        files = list(ds_path.rglob("*.parquet"))
        if not files: continue
        
        # Amostragem para PPM (muitos arquivos/categorias)
        ds_dfs = []
        for f in files:
            df = pd.read_parquet(f)
            # Limpeza SIDRA
            df = df[df['V'] != 'Valor']
            ds_dfs.append(df)
        
        if not ds_dfs: continue
        
        df_ds = pd.concat(ds_dfs, ignore_index=True)
        # Conversão V (Efetivo/Produção)
        df_ds['V'] = pd.to_numeric(df_ds['V'], errors='coerce')
        # Conversão Ano (D3N)
        if 'D3N' in df_ds.columns:
            df_ds['D3N'] = pd.to_numeric(df_ds['D3N'], errors='coerce').astype('Int64')
        
        # Estatísticas de Rebanho por Categoria
        # D4N costuma ser o detalhamento do tipo (ex: Bovinos, Vacas ordenhadas)
        stats = df_ds.groupby(['D3N', 'D2N', 'D4N'])['V'].agg(['count', 'sum', 'mean', 'max']).reset_index()
        stats['categoria_rebanho'] = ds_name
        all_stats.append(stats)
    
    if not all_stats: return None
    
    return pd.concat(all_stats, ignore_index=True)

stats_ppm = analyze_ppm_qualitative(PPM_BRONZE_DIR)

if stats_ppm is not None:
    # Ajustando nomes das colunas para clareza
    stats_ppm.columns = ['Ano', 'Variavel', 'Subtipo_Produto', 'Qtd_Municipios', 'Total_Efetivo', 'Media_por_Mun', 'Maior_Rebanho', 'Categoria']
    
    # Exporta Estatísticas Consolidadas
    output_path = OUTPUT_DIR / "resumo_consolidado_ppm.parquet"
    stats_ppm.to_parquet(output_path, index=False)
    
    print(f"\n✅ Resumo consolidado PPM exportado para: {output_path}")
    print("\n📈 Visão Geral das Categorias (Último Ano):")
    latest_year = stats_ppm['Ano'].max()
    display(stats_ppm[stats_ppm['Ano'] == latest_year].sort_values(by='Total_Efetivo', ascending=False))


🔍 Analisando 12 categorias de rebanhos (PPM)...


Processando Categorias PPM: 100%|██████████| 12/12 [00:00<00:00, 24.81it/s]


✅ Resumo consolidado PPM exportado para: /home/vinicius/Downloads/estudo/fatec/SABADO-TE-ANALISE-DADOS/notebooks/etl-dados-necessarios/ppm/data/resume/resumo_consolidado_ppm.parquet

📈 Visão Geral das Categorias (Último Ano):


,Ano,Variavel,Subtipo_Produto,Qtd_Municipios,Total_Efetivo,Media_por_Mun,Maior_Rebanho,Categoria
15,2024,Efetivo dos rebanhos,Bovino,5537,238180757.0,43016.210403,2519911.0,BOVINOS
3,2024,Efetivo dos rebanhos,,0,0.0,NaN,NaN,CAPRINOS
7,2024,Efetivo dos rebanhos,,0,0.0,NaN,NaN,CODORNAS
11,2024,Efetivo dos rebanhos,,0,0.0,NaN,NaN,GALINHAS
19,2024,Efetivo dos rebanhos,,0,0.0,NaN,NaN,GALINACEOS_TOTAL
23,2024,Efetivo dos rebanhos,,0,0.0,NaN,NaN,ASININOS
27,2024,Efetivo dos rebanhos,,0,0.0,NaN,NaN,OVINOS
31,2024,Efetivo dos rebanhos,,0,0.0,NaN,NaN,EQUINOS
35,2024,Efetivo dos rebanhos,,0,0.0,NaN,NaN,MUAR
39,2024,Efetivo dos rebanhos,,0,0.0,NaN,NaN,SUINOS_MATRIZES
